# Manual Model Evaluation
This notebook allows for manual evaluation of the Global Models (xT and TransGoalNet) using the pre-saved `test_actions.pkl` dataset.

In [1]:
import sys
import os
import pandas as pd
from IPython.display import display, Markdown

# Setup paths to import from src
framework_dir = os.path.dirname(os.getcwd())
if framework_dir not in sys.path:
    sys.path.append(framework_dir)
    sys.path.append(os.path.join(framework_dir, 'src'))

from src.engine.xt_model import ExpectedThreat
from src.engine.transgoalnet import evaluate_transgoalnet
from src.engine.metrics import generate_model_evaluation_report
from src import config

## 1. Load the Test Split
The test split was saved during the Global Model Training phase to drastically reduce memory overhead during evaluation.

In [2]:
if not os.path.exists(config.TEST_ACTIONS_FILE):
    print("Error: Test actions file not found. Ensure `train_all_models.py` has been run.")
else:
    test_actions = pd.read_pickle(config.TEST_ACTIONS_FILE)
    print(f"Loaded {len(test_actions)} test actions across {test_actions['match_id'].nunique()} matches.")

Loaded 615886 test actions across 347 matches.


## 2. Load Models
Load the trained Basic xT model and the TransGoalNet model checkpoints.

In [3]:
xt_checkpoint = config.XT_GLOBAL_CHECKPOINT
trans_checkpoint = config.TGN_GLOBAL_CHECKPOINT

if not os.path.exists(xt_checkpoint) or not os.path.exists(trans_checkpoint):
    print("Error: Model checkpoints not found.")
else:
    print("Loading Basic xT Model...")
    xt_model = ExpectedThreat.load_checkpoint(xt_checkpoint)
    print("Models ready!")

Loading Basic xT Model...
Models ready!


## 3. Run Evaluation
Evaluate TransGoalNet using the loaded dataset and models.

In [4]:
print("Evaluating TransGoalNet...")
eval_metrics_dict = evaluate_transgoalnet(test_actions, xt_model, trans_checkpoint)

# Display the metrics
print("\n--- Evaluation Metrics ---")
for category, met in eval_metrics_dict.items():
    print(f"{category}:")
    print(f"  Metric:  {met.get('Metric')}")
    print(f"  Value:   {met.get('Value'):.6f}")
    print(f"  Meaning: {met.get('Meaning')}\n")

Evaluating TransGoalNet...


D:\MSC DS ABDN\PROJ_The science of football - Jan 2026\CODE\Vibe\football_analytics_framework\src\engine\transgoalnet.py:250: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m


--- Evaluation Metrics ---
Statistical:
  Metric:  Brier Score / MSE
  Value:   0.000166
  Meaning: Is the probability math 'honest'?

Tactical:
  Metric:  Attention Weights (Max focus)
  Value:   0.504106
  Meaning: Is the model looking at the right players/lanes?

Stability:
  Metric:  Mean Absolute Change
  Value:   0.004085
  Meaning: Does the xT value jump around too much?

Success:
  Metric:  Pearson Correlation
  Value:   0.000000
  Meaning: Does 'high xT' actually lead to more points?



In [10]:
eval_progress_file = os.path.join(config.LOGS_DIR, "global_tgn_eval.md")
os.makedirs(config.LOGS_DIR, exist_ok=True)
report_md = generate_model_evaluation_report(eval_metrics_dict, eval_progress_file)

print(f"Evaluation Report saved to `{eval_progress_file}`")

Evaluation Report saved to `D:\MSC DS ABDN\PROJ_The science of football - Jan 2026\CODE\Vibe\football_analytics_framework\logs\global_tgn_eval.md`
